# Fine-tune Laya for Dhristi's fast path (Kaggle GPU)

Runs `scripts/laya-finetune/train_cpu.py` from the `sih-2026` repo on a GPU with the Laya authors' recipe: the whole encoder, 4 epochs, `max_len` 1024 / `head_max_len` 256. On 30 September 2026 two CPU runs that trained only the top 6 of 28 layers underfit (29 of 120 training steps right, base model 24).

**Kaggle settings:** Accelerator `GPU T4 x2` (one GPU is used) or `P100`; Internet `On`.

The data is synthetic (generated by `gen_data.py`, disjoint from the evaluation cases). A good score shows the model learned this task's format on synthetic pages, not that it works on real sites.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "laya==0.3.22" safetensors huggingface_hub
import torch; print("CUDA:", torch.cuda.is_available())

In [ ]:
BRANCH = "claude/determined-babbage-mhi2zj"   # change to master once PR #41 is merged
!rm -rf sih-2026 && git clone -q --depth 1 --branch $BRANCH https://github.com/francisreubenr-rvu/sih-2026
%cd sih-2026
!git log --oneline -1

In [ ]:
from huggingface_hub import snapshot_download
BASE = snapshot_download("convaiinnovations/laya", allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"])
print(BASE)

## Data: 3,000 steps, same generator and seed as the committed 900-step set

In [ ]:
!python scripts/laya-finetune/gen_data.py --n 3000 --seed 20260930 --out /kaggle/working/train3000.jsonl

## Train (whole encoder, 4 epochs, 1024 / 256)

In [ ]:
!EPOCHS=4 TRAIN_TOP_LAYERS=28 MAX_LEN=1024 HEAD_MAX_LEN=256 python scripts/laya-finetune/train_cpu.py $BASE /kaggle/working/train3000.jsonl /kaggle/working/laya-dhristi-gpu

## Does it fit its training data? (the check both CPU runs failed)

In [ ]:
!python scripts/laya-finetune/diagnose.py /kaggle/working/laya-dhristi-gpu $BASE

## Accuracy on the design and held-out cases (LLM answers reused from the committed run; no Groq key needed)

In [ ]:
!WARDEN_LAYA_DEVICE=cuda python scripts/cloud-models/fastpath_bench.py --backends laya --laya-model /kaggle/working/laya-dhristi-gpu \
    --reps 1 --llm-from Benchmarks/results/fastpath-bench-v01.json --out /kaggle/working/fastpath-bench-laya-gpu.json

## Keep the weights

Download `/kaggle/working/laya-dhristi-gpu.zip` from the Output panel, or push it to a Hugging Face repo with an `HF_TOKEN` Kaggle secret. Commit `fastpath-bench-laya-gpu.json` back to `Benchmarks/results/`.

In [ ]:
!cd /kaggle/working && zip -qr laya-dhristi-gpu.zip laya-dhristi-gpu && ls -lh laya-dhristi-gpu.zip